In [24]:
from docx import Document
import re
import os

In [ ]:
from docx import Document
import re
horizontal_lines = "______________________________________________________________________________________________________________________________________________"


print(len(horizontal_lines))
def extract_amharic_text(docx_path):
    doc = Document(docx_path)
    lines = [para.text.strip() for para in doc.paragraphs if para.text.strip()]
    amharic_texts = []
    capturing = False
    
    for line in lines:
        if re.match(r'[_]+', line):  # Detect horizontal lines
            if capturing:
                break  # Stop capturing after the second line
            capturing = True
            continue
        
        if capturing:
            amharic_texts.append(line)
    
    return '\n'.join(amharic_texts)

# Replace with the actual file path
docx_path = "all_forms_new/2260.docx"
bad_ids = []
empty = 0
long_sentences= []
for i, file in enumerate(os.listdir("all_forms_new")):
    try:
        amharic_text = extract_amharic_text("all_forms_new/"+file)[:-142].strip("\n")
        if len(amharic_text)>700:
            long_sentences.append(amharic_text) 
            print("more than 650 ", len(amharic_text))
            bad_ids.append(i+1)
    except Exception as e:
        empty +=1
        print(str(e))

    
print("empty forms", empty)
print(bad_ids)

# amharic_text

In [57]:
new_long_sentences = []

for long in long_sentences:
    for sent in (long.split("።")):
        if sent != "":
            new_long_sentences.append(sent)

In [ ]:
len(new_long_sentences)

In [59]:
def group_sentences_minimized(sentences, max_chars_per_group=650):
    """
    Groups a list of sentences into groups with a maximum character limit,
    minimizing the number of groups. If a sentence is longer than the limit,
    it is broken into smaller sentences without splitting words.

    Args:
      sentences: A list of strings, where each string is a sentence.
      max_chars_per_group: The maximum number of characters allowed in each group.

    Returns:
      A list of lists, where each inner list contains sentences that belong to a group.
    """

    all_fragments = []
    for sentence in sentences:
        if len(sentence) > max_chars_per_group:
            words = sentence.split()
            sub_sentence = ""
            for word in words:
                if len(sub_sentence) + len(word) + 1 <= max_chars_per_group:
                    if sub_sentence:
                        sub_sentence += " "
                    sub_sentence += word
                else:
                    all_fragments.append(sub_sentence)
                    sub_sentence = word
            all_fragments.append(sub_sentence)
        else:
            all_fragments.append(sentence)

    all_fragments.sort(key=len)  # Sort fragments by length

    groups = []
    current_group = []
    current_group_length = 0

    for fragment in all_fragments:
        if current_group_length + len(fragment) <= max_chars_per_group:
            current_group.append(fragment)
            current_group_length += len(fragment)
        else:
            groups.append(current_group)
            current_group = [fragment]
            current_group_length = len(fragment)

    if current_group:
        groups.append(current_group)

    return groups

grouped_sentences = group_sentences_minimized(new_long_sentences)

In [ ]:
len(grouped_sentences)

In [ ]:
for sent in grouped_sentences:
    total_chars = 0
    for sen in sent:
        total_chars += len(sen)
    print(total_chars)


In [63]:
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
import os
def read_first_n_lines(file_path: str, n: int) -> list:
    """Read the first n lines from a file."""
    with open(file_path, 'r', encoding='utf-8') as file:
        return [next(file).strip() for _ in range(n)]

def create_all_forms(file_path, sentences_per_file  = 6, max_chars = 800):
    # filelist = [ f for f in os.listdir("all_forms") if f.endswith(".docx") ]
    # for f in filelist:
    #     os.remove(os.path.join("all_forms", f))
    with open(file_path, 'r', encoding='utf-8') as file:
        current = []
        all_lines = 0
        file_number =  1
        excessive_lines = []

        for i in range(24000):
            current.append(next(file).strip())
            if (i>1) and (i % sentences_per_file) == 0:
                orig_current= current.copy()
                total_chars =sum([len(sent) for sent in current])
                while total_chars > max_chars:
                    print("bad..." ,total_chars)
                    excessive_lines.append(current.pop())
                    total_chars =sum([len(sent) for sent in current])
                # if total_chars == 0:
                #     current = orig_current[0]

                    
                if total_chars > 0:


                    create_single_page_form(current, file_number, "all_forms/"+ str(file_number) +".docx")
                    total_chars =sum([len(sent) for sent in current])

                    print("Single-page form created as ", "all_forms/"+ str(file_number) +".docx", " sentences ", len(current), "chars ",total_chars )
                    all_lines += len(current)
                    file_number+=1
                    current = []
                
                    
        for current in excessive_lines:
            current = [current]
            create_single_page_form(current, file_number, "all_forms/"+ str(file_number) +".docx")
            total_chars =sum([len(sent) for sent in current])

            print("Single-page form created as ", "all_forms/"+ str(file_number) +".docx", " sentences ", len(current), "chars ",total_chars )
            all_lines += len(current)
            file_number+=1            
        print("Number of sentences used", all_lines)

def create_single_page_form(sentences: list, form_number: int, output_path: str) -> None:
    doc = Document()

    # Set margins
    orig_margin = 1.27
    section = doc.sections[0]
    section.top_margin = Cm(orig_margin/4.0)
    section.bottom_margin = Cm(orig_margin/4.0)
    section.left_margin = Cm(orig_margin/3.0)
    section.right_margin = Cm(orig_margin/3.0)

    # Add form number
    form_num = doc.add_paragraph()
    form_num.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT  # Center the form number
    form_num.add_run(f"{form_number:04d}")

    form_num.alignment = WD_PARAGRAPH_ALIGNMENT.RIGHT  # Center the form number
    form_num.add_run("                                                                Writer Number: " + "_" * 20)

    form_num.paragraph_format.space_after = Pt(6)
    form_num.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Center the form number


    # First horizontal line
    doc.add_paragraph("_" * 142)

    # Content paragraph
    para = doc.add_paragraph()
    para.paragraph_format.space_after = Pt(12)

    for i, sentence in enumerate(sentences):
        # para.add_run(sentence.strip("::") + "።")  # Add Ethiopian period after stripping ::
        para.add_run(sentence.strip("::"))  # Add Ethiopian period after stripping ::

        if i < len(sentences) - 1:
            para.add_run(" ")
    para.add_run("\n"+"_" * 142)

    # Middle horizontal line
    # doc.add_paragraph("_" * 105)

    # Add enough empty paragraphs to push content but avoid extra pages
    # for _ in range(8):  # Reduced from 10 to leave more space for footer
    for _ in range(12):  # Reduced from 10 to leave more space for footer

        doc.add_paragraph("\n")

    # Add footer with increased spacing
    footer = section.footer.paragraphs[0] if section.footer.paragraphs else section.footer.add_paragraph()
    footer.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

    # Add the line with extra spacing
    footer_line = footer.add_run("_" * 98)
    footer_line.font.size = Pt(12)

    # Add two line breaks for spacing
    footer.add_run("\n\n")

    # Add the writer number
    # footer.add_run("Writer Number: " + "_" * 20)

    # Save only the first page
    doc.save(output_path)


# Test function with file input
try:
    test_sentences = read_first_n_lines('both_selected_amharic_sentences.txt', 6)
    create_single_page_form(test_sentences, 1, 'test_form.docx')
    print("Single-page form created as 'test_form.docx'")
except FileNotFoundError:
    print("Error: Could not find 'selected_amharic_sentences.txt'")
except Exception as e:
    print(f"An error occurred: {e}")

Single-page form created as 'test_form.docx'


In [64]:
file_number = 4048
for  forms in grouped_sentences:
    create_single_page_form(forms,file_number, "tunga_edits/"+ str(file_number) +".docx")
    file_number +=1